# AU Case Study - Statistical Evaluation Overview (Read-Only)

> Data source: `013_statistical_evaluation.py`; specification = `docs/B0_design_decisions.md`
> plus the final sign-off notes; statistical protocol follows the UK's
> `031_exp_r216_robust_stats.py` (tools imported from the shared `revision_statistics`
> module). This notebook is **read-only display only, and writes no artifacts**; all
> numbers come from saved files under `data/processed/evaluation/` (nothing is hand-typed).
>
> **Protocol**: inference unit = SA4 x 12 (corrected per the actual usable-station count)
> -> seed-averaged -> 12-region paired difference -> **2^12 = 4096 exact sign-flip
> permutations** (minimum attainable p = 2/4096 ~= 4.88e-4) + region-level paired
> bootstrap B=10^4 (outer layer only, CI is not back-derived into p) + **Holm correction
> per metric family** (family = 12 comparisons); static-vs-static comparisons use the
> no-seed branch; per-seed permutation p-values are saved in full + a mixedlm robustness
> arm + Moran's I diagnostics (queen-contiguity primary W + kNN(3) supplementary).
>
> **Sensitivity arms**: (1) SA3 x 34 re-aggregation (Monte-Carlo sign-flip, B=10^5, fixed
> seed; the corr arm only for units with >= 3 stations); (2) exclusion of the 5 stations
> flagged by day/night ratio (pure statistical-layer station-set filter).
>
> **Deviation record** (full detail in `evaluation_summary.json.meta.deviations`): SA3 unit
> count = 34 (corrected per the actual usable-station count, vs. an earlier estimate of 35);
> SA3 Monte-Carlo uses B=10^5 (a stricter choice than the ">= 10^5" specification).

In [ ]:
%matplotlib inline
# Environment and artifact paths (read-only)
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 160)

AU_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL = AU_DIR / "data" / "processed" / "evaluation"

hyp = pd.read_csv(EVAL / "au_hypothesis_tests.csv", encoding="utf-8-sig")
direction = pd.read_csv(EVAL / "au_vs_uk_direction.csv", encoding="utf-8-sig")
three = pd.read_csv(EVAL / "three_case_comparison.csv", encoding="utf-8-sig")
summary = json.loads((EVAL / "evaluation_summary.json").read_text(encoding="utf-8"))
registry = pd.read_csv(EVAL / "au_comparison_registry.csv", encoding="utf-8-sig")

main = hyp[hyp["branch"] == "main_sa4"].copy()
print(f"Comparison registry: {len(registry)} comparisons | Test rows: {len(hyp)} (3 branches x 12 comparisons x 3 metrics)")

## 1. Core Hypothesis Test Table (Main Branch: SA4 x 12, Exact Permutation + Holm)

Comparison list (`au_comparison_registry.csv`, saved to disk in advance):
- **H-AU1** multiplicative correction harms the GNN base: GNNpostNP/N/P vs GNN + GNNpostNP
  vs GNNpostP (the core antagonism finding from the UK case study);
- **H-AU2** additive correction repairs it: GNNaddNP vs GNNpostNP, GNNaddP vs GNN;
- **H-AU3** static bases benefit from correction: UniNP vs Uni, GPMpostNP vs GPM,
  GPMpostNP vs GPMpostP;
- **Cross-base**: GNNaddP / GNNaddNP vs GPMpostNP (per the UK comparison list);
- **prior channel**: GNNpriorNP vs GNN (the ntl_prox training arm).

Delta = A - B (negative rmse/mae = A is better; positive corr = A is better).

In [ ]:
cols = ["comparison_id", "comparison", "hypothesis", "metric", "mean_diff",
        "perm_p", "holm_p", "sig", "ci_lo", "ci_hi", "spatial_warning"]
tbl = main[cols].copy()
tbl.columns = ["#", "Comparison", "Hypothesis", "Metric", "Delta(A-B)", "Perm p", "Holm p",
               "Sig", "CI lo", "CI hi", "Spatial warning"]
with pd.option_context("display.float_format", "{:.4g}".format):
    display(tbl.set_index(["#", "Comparison", "Hypothesis", "Metric"]))

## 2. Core Readout (RMSE Primary Metric, Conclusion Fields Generated by Rule)

In [ ]:
for hyp_key in ["h_au1_multiplicative_on_gnn", "h_au2_additive_repair",
                "h_au3_static_base_gain", "cross_base", "prior_channel"]:
    blk = summary[hyp_key]
    print("=" * 100)
    if isinstance(blk, dict) and "hypothesis" in blk:
        print(f"[{hyp_key}] {blk['hypothesis']}")
        comps = blk["comparisons"]
    else:
        print(f"[{hyp_key}]")
        comps = blk
    for k, v in comps.items():
        print("  " + v["conclusion"])
print("=" * 100)
print(f"H-AU1 multiplicative antagonism (harms the GNN base) replicated in AU: "
      f"{summary['h_au1_multiplicative_on_gnn']['replicated_harm']}")
print(f"H-AU2 additive beats multiplicative: "
      f"{summary['h_au2_additive_repair']['additive_beats_multiplicative']} | "
      f"additive beats base: {summary['h_au2_additive_repair']['additive_improves_base']}")
print(f"H-AU3 both static bases gain significantly: "
      f"{summary['h_au3_static_base_gain']['both_static_gains_significant']}")

## 3. Direction Consistency with UK

`direction_match_uk` = sign(AU Delta) == sign(UK Delta). UK reference values are read-only
citations of `exp_r12/full_matrix.csv` (Delta) and `exp_r216/robust_tests.csv` /
`exp_r12/new_comparisons.csv` (frozen p-values); the UK counterpart arm for the prior
channel comes from `exp0_kfold_prior` (supplementary, read-only, anchored to exp_r216 #8).

In [ ]:
dcols = ["comparison_id", "comparison", "metric", "au_delta", "au_sig",
         "uk_delta", "uk_sig", "direction_match_uk", "uk_source"]
dt = direction[dcols].copy()
dt.columns = ["#", "Comparison", "Metric", "AU Delta", "AU Sig", "UK Delta", "UK Sig",
              "Direction match", "UK source"]
with pd.option_context("display.float_format", "{:.4g}".format,
                       "display.max_colwidth", 60):
    display(dt.set_index(["#", "Comparison", "Metric"]))
cons = summary["uk_direction_consistency"]
print(f"Direction-match count: {cons['n_direction_match']}/{cons['n_rows_with_uk_delta']}"
      f"(rmse {cons['per_metric_match']['rmse']}/{cons['per_metric_total']['rmse']}, "
      f"mae {cons['per_metric_match']['mae']}/{cons['per_metric_total']['mae']}, "
      f"corr {cons['per_metric_match']['corr']}/{cons['per_metric_total']['corr']})")

In [ ]:
# Direction-consistency plot: comparison x metric matrix (green = match, red = mismatch)
from matplotlib.colors import ListedColormap

metrics = ["rmse", "mae", "corr"]
comps = registry["comparison"].tolist()
mat = np.full((len(comps), len(metrics)), np.nan)
for i, comp in enumerate(comps):
    for j, m in enumerate(metrics):
        row = direction[(direction["comparison"] == comp)
                        & (direction["metric"] == m)]
        v = row["direction_match_uk"].iloc[0]
        if pd.notna(v):
            mat[i, j] = 1.0 if bool(v) else 0.0

fig, ax = plt.subplots(figsize=(7, 7.5))
cmap = ListedColormap(["#c0392b", "#27ae60"])
ax.imshow(np.ma.masked_invalid(mat), cmap=cmap, vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(metrics)), [m.upper() for m in metrics])
ax.set_yticks(range(len(comps)), comps)
for i in range(len(comps)):
    for j in range(len(metrics)):
        if np.isnan(mat[i, j]):
            ax.text(j, i, "NA", ha="center", va="center", color="grey")
        else:
            au_sig = direction[(direction["comparison"] == comps[i])
                               & (direction["metric"] == metrics[j])]["au_sig"].iloc[0]
            ax.text(j, i, ("match" if mat[i, j] == 1 else "mismatch")
                    + ("*" if bool(au_sig) else ""),
                    ha="center", va="center", color="white", fontsize=9)
ax.set_title("AU vs UK mechanism direction consistency (* = significant on the AU side, Holm-corrected)")
plt.tight_layout()
plt.show()

## 4. Three-Case Comparison Table (UK / DE / AU side by side, mean RMSE per arm)

- UK = 16-region mean from `exp_r12/full_matrix.csv` (GNN arm is a 3-seed average);
- DE = `Germany/results/static_allocation/boerde_static_metrics.csv` (**single-region
  Börde**; the NP arm uses the sequential-stacking convention; the additive arm has no
  frozen result yet; **the GNN arm is still training and to be filled in**);
- AU = 12-region mean from the static baseline table + k-fold GNN results (3-seed average).
- The three cases use different demand conventions/units (MVA / MW); **cross-case
  comparisons are limited to mechanism direction, not absolute values**.

In [ ]:
tcols = ["arm", "base", "form", "signal", "uk_rmse_mean_16regions",
         "de_rmse_boerde", "de_status", "au_rmse_mean_12regions"]
tt = three[tcols].copy()
tt.columns = ["Arm", "Base", "Form", "Signal", "UK RMSE (16-region mean)",
              "DE RMSE (Börde)", "DE status", "AU RMSE (12-region mean)"]
with pd.option_context("display.float_format", "{:.3f}".format):
    display(tt.set_index("Arm"))

In [ ]:
# Cross-case mechanism-direction visualization: each arm's RMSE relative to its own Uniform base (unit-independent)
uk_uni = three.loc[three["arm"] == "Uni", "uk_rmse_mean_16regions"].iloc[0]
de_uni = three.loc[three["arm"] == "Uni", "de_rmse_boerde"].iloc[0]
au_uni = three.loc[three["arm"] == "Uni", "au_rmse_mean_12regions"].iloc[0]
plot_arms = ["Uni", "UniNP", "GPM", "GPMpostP", "GPMpostNP",
             "GNN", "GNNpostP", "GNNpostNP", "GNNaddP", "GNNaddNP"]
sub = three.set_index("arm").loc[plot_arms]
x = np.arange(len(plot_arms))
w = 0.27
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x - w, sub["uk_rmse_mean_16regions"] / uk_uni, w, label="UK (16 regions)")
ax.bar(x, sub["de_rmse_boerde"] / de_uni, w, label="DE (Börde; GNN pending)")
ax.bar(x + w, sub["au_rmse_mean_12regions"] / au_uni, w, label="AU (12 regions)")
ax.axhline(1.0, color="grey", lw=0.8, ls="--")
ax.set_xticks(x, plot_arms, rotation=30, ha="right")
ax.set_ylabel("RMSE / own Uni base")
ax.set_title("Cross-case mechanism-direction comparison (relative to each case's own Uniform base; <1 = better than uniform allocation)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sensitivity Arms (SA3 x 34 Re-aggregation / PV-Station Exclusion)

Flip fields are generated by numeric rule: `sig_flipped_vs_main` = sensitivity-arm
significance != main branch; `direction_flipped_vs_main` = sensitivity-arm Delta sign
!= main branch.

In [ ]:
for branch, name in [("sa3_sensitivity", "SA3 x 34 re-aggregation (Monte-Carlo B=10^5)"),
                     ("pv_excl", "PV-station exclusion, 5 stations (exact 2^12)")]:
    sub = hyp[hyp["branch"] == branch]
    print(f"-- {name} --")
    print(f"  Significance flips: {int(sub['sig_flipped_vs_main'].sum())}/{len(sub)} | "
          f"Direction flips: {int(sub['direction_flipped_vs_main'].sum())}/{len(sub)}")
    fl = sub[sub["sig_flipped_vs_main"] | sub["direction_flipped_vs_main"]]
    if len(fl):
        with pd.option_context("display.float_format", "{:.4g}".format):
            display(fl[["comparison", "metric", "n_units", "mean_diff", "perm_p",
                        "holm_p", "sig", "sig_flipped_vs_main",
                        "direction_flipped_vs_main"]].set_index(["comparison", "metric"]))
    else:
        print("  (no flipped rows)")
sa3s = summary["sa3_sensitivity"]
print(f"SA3 sensitivity-arm units: rmse/mae {sa3s['n_units_rmse_mae']} | "
      f"corr {sa3s['n_units_corr']} ({sa3s['corr_unit_rule']})")

## 6. Diagnostics and Anchoring

- **Moran's I** (diagnostic, not a hypothesis test): queen-contiguity primary W (SA4
  dissolve, with a no-zero-rows assertion) + kNN(3) supplementary W (centroid,
  EPSG:7856 -- see deviation record item 3); significant rows are reported alongside
  their spatial-block permutation replacement counts;
- **mixedlm** (robustness arm): region x seed crossed random intercepts;
- **Re-aggregation anchor**: GNN-arm per-SA4 metrics vs. the frozen k-fold csv (4 decimal
  places, half-ulp tolerance); static arms vs. `au_static_metrics.csv` (1e-9) -- the
  evaluation script hard-fails if the anchor check does not pass.

In [ ]:
print(f"Moran queen spatial warnings: {summary['moran_diagnostics']['n_spatial_warning_queen']}"
      f"/{summary['moran_diagnostics']['n_tests']}")
print(f"mixedlm: converged {summary['mixedlm']['n_converged']}/{summary['mixedlm']['n_models']}, "
      f"agrees with permutation test at 0.05: {summary['mixedlm']['n_agree_with_perm_at_0.05']}")
anchor = summary["reaggregation_anchor"]
print(f"Re-aggregation anchor: GNN max deviation {anchor['gnn_max_abs_dev']:.3e} (tolerance 5e-5) | "
      f"static {max(anchor['static_max_abs_dev'].values()):.3e} (tolerance 1e-9) | "
      f"passed = {anchor['passed']}")
print()
print("Deviation record (meta.deviations):")
for d in summary["meta"]["deviations"]:
    print(f"  [{d['id']}] {d['item']} -- {d['detail']}")